# Baseline Feature Workflow

이 노트북은 공개 동일 종 음원으로 만든 1차 기준선 파이프라인을 한 번에 점검하기 위한 실험 노트북이다.

흐름:
1. 매니페스트 확인
2. 특성추출 산출물 확인
3. 기준선 리포트 요약
4. confusion matrix 확인
5. 단일 파일 예측 결과 확인

In [ ]:
from pathlib import Path
import json
import csv
import numpy as np
import pandas as pd
from IPython.display import display, Image

ROOT = Path.cwd().resolve().parent.parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
ROOT

In [ ]:
manifest_path = ROOT / 'data-workspace' / 'metadata' / 'inat_candidate_manifest.csv'
manifest_df = pd.read_csv(manifest_path)
manifest_df.head()

In [ ]:
manifest_df.groupby('species_code').size().sort_index()

In [ ]:
features_npz = ROOT / 'ml-workspace' / 'artifacts' / 'features' / 'inat_candidate_features.npz'
features_meta = ROOT / 'ml-workspace' / 'artifacts' / 'features' / 'inat_candidate_features_metadata.csv'
report_path = ROOT / 'ml-workspace' / 'reports' / 'inat_candidate_feature_report.json'
assets_dir = ROOT / 'ml-workspace' / 'reports' / 'assets'

features_blob = np.load(features_npz, allow_pickle=True)
features_blob.files

In [ ]:
X = features_blob['X']
print('shape:', X.shape)
print('feature_version:', features_blob['feature_version'])
print('window_sec:', features_blob['window_sec'])
print('hop_sec:', features_blob['hop_sec'])

In [ ]:
feature_meta_df = pd.read_csv(features_meta)
feature_meta_df.head()

In [ ]:
feature_meta_df.groupby('species_code').size().sort_index()

In [ ]:
report = json.loads(report_path.read_text(encoding='utf-8'))
summary = {
    'accuracy': report['metrics']['accuracy'],
    'macro_avg': report['metrics']['macro avg'],
    'weighted_avg': report['metrics']['weighted avg'],
}
summary

In [ ]:
per_class = pd.DataFrame({
    label: report['metrics'][label]
    for label in report['dataset_summary']['classes']
}).T[['precision', 'recall', 'f1-score', 'support']]
per_class

In [ ]:
pd.DataFrame(report['confusion_matrix']['off_diagonal']).head(10)

In [ ]:
display(Image(filename=str(assets_dir / 'confusion_matrix.png')))
display(Image(filename=str(assets_dir / 'per_class_f1.png')))

In [ ]:
sample_prediction_path = ROOT / 'ml-workspace' / 'reports' / 'sample_prediction_chammaemi.json'
sample_prediction = json.loads(sample_prediction_path.read_text(encoding='utf-8'))
pd.DataFrame(sample_prediction['aggregate_ranking']).head(3)

## 재실행 명령

터미널에서 아래 순서로 다시 만들 수 있다.

```powershell
.\.venv\Scripts\python.exe .\scripts\extract_manifest_features.py .\data-workspace\metadata\inat_candidate_manifest.csv --features-out .\ml-workspace\artifacts\features\inat_candidate_features.npz --metadata-out .\ml-workspace\artifacts\features\inat_candidate_features_metadata.csv
.\.venv\Scripts\python.exe .\scripts\train_baseline_from_features.py .\ml-workspace\artifacts\features\inat_candidate_features.npz .\ml-workspace\artifacts\features\inat_candidate_features_metadata.csv --report-out .\ml-workspace\reports\inat_candidate_feature_report.json --model-out .\ml-workspace\artifacts\inat_candidate_feature_model.joblib
.\.venv\Scripts\python.exe .\scripts\render_baseline_report_assets.py .\ml-workspace\reports\inat_candidate_feature_report.json --out-dir .\ml-workspace\reports\assets
```